In [16]:
import torch

from tqdm import tqdm

from stock_mpt import StockMPT, LinearModel
from stock_mpte import StockMPTE, ensemble_uncertainty_analysis
from dataloader_builder_mpt import build_dataloaders
from model_training_mpt import precision_recall_curve
from setup import StockMPT_cfg as cfg, LinearModel_cfg as linear_cfg
from setup import path_data_preprocessor
from setup import ID, STEP, RV_THRESH, FILE_LIMIT

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


In [17]:
print(path_data_preprocessor)
dls, train_norms = build_dataloaders(path_data_preprocessor)

counts = torch.zeros(3, dtype=torch.long)

for _, y in dls["train"]:
    counts += torch.bincount(y.flatten().cpu(), minlength=3)

print("Counts:", counts)
print("Distribution:", counts / counts.sum())

preprocessed_data/data_1min_2021_2026_4
Building DataLoaders...
Train dataset samples: 32,361
Train loader batches:  252
Batch size:            128
Counts: tensor([3500113, 5328931, 3589516])
Distribution: tensor([0.2818, 0.4291, 0.2890])


In [22]:
path1 = f"model_parameters/best_stock_mpt_v{ID}{STEP}-{RV_THRESH}-5-{FILE_LIMIT}-s1"
path2 = f"model_parameters/best_stock_mpt_v{ID}{STEP}-{RV_THRESH}-5-{FILE_LIMIT}-s2"
path3 = f"model_parameters/best_linear_model_v{ID}{STEP}-{RV_THRESH}-5-{FILE_LIMIT}"
model1 = StockMPT(cfg, train_norms).to(device)
model2 = StockMPT(cfg, train_norms).to(device)
model3 = LinearModel(linear_cfg, train_norms).to(device)

model1.load_state_dict(torch.load(path1, map_location=device)["model"])
model2.load_state_dict(torch.load(path2, map_location=device)["model"])
model3.load_state_dict(torch.load(path3, map_location=device)["model"])

stockMPTE = StockMPTE([model1, model2, model3])

In [23]:
precision_recall_curve(dls["val"], stockMPTE, device, cls=2)
print("\n")
precision_recall_curve(dls["val"], stockMPTE, device, cls=1)
print("\n")
precision_recall_curve(dls["val"], stockMPTE, device, cls=0)

0.10 | PREC 0.3078 | REC 0.9946 | N 3871501
0.15 | PREC 0.3292 | REC 0.9658 | N 3514015
0.20 | PREC 0.3539 | REC 0.9095 | N 3078460
0.25 | PREC 0.3790 | REC 0.8235 | N 2603177
0.30 | PREC 0.4054 | REC 0.6975 | N 2061138
0.35 | PREC 0.4347 | REC 0.5198 | N 1432745
0.40 | PREC 0.4703 | REC 0.2947 | N 750663
0.45 | PREC 0.5361 | REC 0.0812 | N 181436
0.50 | PREC 0.6466 | REC 0.0142 | N 26276
0.55 | PREC 0.7295 | REC 0.0027 | N 4510
0.60 | PREC 0.8140 | REC 0.0004 | N 543
0.65 | PREC 0.8000 | REC 0.0000 | N 25
0.70 | PREC 0.0000 | REC 0.0000 | N 0
0.75 | PREC 0.0000 | REC 0.0000 | N 0
0.80 | PREC 0.0000 | REC 0.0000 | N 0
0.85 | PREC 0.0000 | REC 0.0000 | N 0
0.90 | PREC 0.0000 | REC 0.0000 | N 0
0.95 | PREC 0.0000 | REC 0.0000 | N 0


0.10 | PREC 0.4266 | REC 0.9918 | N 3759944
0.15 | PREC 0.4491 | REC 0.9765 | N 3516358
0.20 | PREC 0.4781 | REC 0.9497 | N 3212269
0.25 | PREC 0.5108 | REC 0.9118 | N 2886910
0.30 | PREC 0.5451 | REC 0.8630 | N 2560391
0.35 | PREC 0.5808 | REC 0.8049 | N 22

In [6]:
ensemble_uncertainty_analysis(
    dls["val"],
    stockMPTE,
    device,
    cls=2,
    prob_threshold=0.50
)

Signals: 134480
Precision: 0.5705
STD <= 0.010 | PREC 0.5648 | N 18554
STD <= 0.020 | PREC 0.5651 | N 56926
STD <= 0.030 | PREC 0.5683 | N 90394
STD <= 0.050 | PREC 0.5703 | N 124110
STD <= 0.075 | PREC 0.5704 | N 133304
STD <= 0.100 | PREC 0.5705 | N 134343
STD <= 0.150 | PREC 0.5705 | N 134477


In [27]:
handpicked_dls, train_norms = build_dataloaders("handpicked_data", False, drop_last = False)

Building DataLoaders...


In [28]:
precision_recall_curve(handpicked_dls["test"], stockMPTE, device, cls=2)
print("\n")
precision_recall_curve(handpicked_dls["test"], stockMPTE, device, cls=1)
print("\n")
precision_recall_curve(handpicked_dls["test"], stockMPTE, device, cls=0)

0.10 | PREC 0.3065 | REC 1.0000 | N 1155
0.15 | PREC 0.3065 | REC 1.0000 | N 1155
0.20 | PREC 0.3065 | REC 1.0000 | N 1155
0.25 | PREC 0.2538 | REC 0.6638 | N 926
0.30 | PREC 0.2409 | REC 0.5763 | N 847
0.35 | PREC 0.2340 | REC 0.5254 | N 795
0.40 | PREC 0.2213 | REC 0.4576 | N 732
0.45 | PREC 0.1818 | REC 0.0056 | N 11
0.50 | PREC 0.0000 | REC 0.0000 | N 0
0.55 | PREC 0.0000 | REC 0.0000 | N 0
0.60 | PREC 0.0000 | REC 0.0000 | N 0
0.65 | PREC 0.0000 | REC 0.0000 | N 0
0.70 | PREC 0.0000 | REC 0.0000 | N 0
0.75 | PREC 0.0000 | REC 0.0000 | N 0
0.80 | PREC 0.0000 | REC 0.0000 | N 0
0.85 | PREC 0.0000 | REC 0.0000 | N 0
0.90 | PREC 0.0000 | REC 0.0000 | N 0
0.95 | PREC 0.0000 | REC 0.0000 | N 0


0.10 | PREC 0.4076 | REC 0.8879 | N 952
0.15 | PREC 0.1515 | REC 0.1373 | N 396
0.20 | PREC 0.1499 | REC 0.1327 | N 387
0.25 | PREC 0.1489 | REC 0.1281 | N 376
0.30 | PREC 0.1519 | REC 0.1259 | N 362
0.35 | PREC 0.1512 | REC 0.1190 | N 344
0.40 | PREC 0.1419 | REC 0.1007 | N 310
0.45 | PREC 0.14

In [29]:
ensemble_uncertainty_analysis(
    handpicked_dls["test"],
    stockMPTE,
    device,
    cls=2,
    prob_threshold=0.40
)

Signals: 732
Precision: 0.2213
STD <= 0.030 | PREC 0.0000 | N 1
STD <= 0.050 | PREC 0.0000 | N 2
STD <= 0.075 | PREC 0.3333 | N 3
STD <= 0.100 | PREC 0.2249 | N 289
STD <= 0.150 | PREC 0.2228 | N 727
